In [10]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import random

from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# plot correlation
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr

# Set publication-quality style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 12,
    'axes.linewidth': 1.5,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12,
    'axes.titlesize': 16,
    'pdf.fonttype': 42
})

# Set random seed for reproducibility
seed = 0
random.seed(seed)
np.random.seed(seed)    
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

GX = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/241126_gene_expression_all.csv')
# GX = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/241126_gene_expression_RBP_only.csv')
PSI = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/250407_measurement_ratio.csv')
Barcode = pd.read_csv('~/lab_projects/splicing/for_kai/V4_data_with_offset_major_event_only/250407_twist_barcodes_with_splice_site_annotation.csv')
index_offset_shortlist = pd.read_csv("~/Downloads/shortlist_satmut_parent_seq.csv")

GX_dict = GX.groupby("condition").apply(
    lambda x: np.array(x.drop(columns=["Unnamed: 0", "condition"]).values.flatten().tolist())
).to_dict()

/tmp/ipykernel_193827/4209658503.py:49: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  GX_dict = GX.groupby("condition").apply(


In [11]:
# make index as the barcode
Barcode = Barcode.set_index("index_offset")
Barcode = Barcode.drop(columns=["Unnamed: 0"])

# Filter the barcodes that are in index_offset_shortlist by ExonID, matching ExonID prefix in index_offset
# The ExonID in index_offset_shortlist is like "ID", while Barcode index is like "ID__000"
exonid_shortlist = set(index_offset_shortlist['ExonID'].astype(str))
# Keep only barcodes whose index starts with an ExonID in the shortlist
Barcode_filtered = Barcode[Barcode.index.to_series().str.split("__").str[0].isin(exonid_shortlist)]
Barcode_filtered

,padded_sequence,exon_intron_map,is_splice_acceptor,is_splice_donor
index_offset,,,,
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0,CTTCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000089006.17;SNX5;chr20-17944121-17944144-17943109-17943195-17947485-17947621__0:0:0,GCTTCGACACCGAGCTCGCTCAAAATTTATTCCTTGAGTTTAGCTG...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000104695.13;PPP2CB;chr8-30799716-30799755-30797580-30797754-30812319-30812597__0:0:0,GCTTCGACACCGAGCTCGGCTTTTATAAGGCACTTTTCAAGTGTTT...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000112818.11;MEP1A;chr6-46793551-46793576-46793388-46793467-46793665-46793716__0:0:0,CTGAGATTGAATCCAGGAAATGAAGCTTCGACACCGAGCTCGTGGA...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000116001.17;TIA1;chr2-70229058-70229091-70227734-70227822-70229263-70229318__0:0:0,GCTTCGACACCGAGCTCGCTATAAACATCATACATGTTTGCAGTAA...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000118855.21;MFSD1;chr3-158823427-158823525-158821983-158822140-158824123-158824236__0:0:0,CTTCGACACCGAGCTCGTTGCTAATCTGCATTAAAGGAAAATCACC...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000119321.9;FKBP15;chr9-113202960-113203035-113202530-113202629-113206508-113206578__0:0:0,GCTTCGACACCGAGCTCGGCTGATTTTATATTGCTGATTTTATATT...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000124549.14;BTN2A3P;chr6-26422088-26422197-26421390-26421624-26422932-26423759__0:0:0,GCTTCGACACCGAGCTCGCGGGAGAGGTTGGGCCCGACCAGCACTG...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...
ENSG00000126777.18;KTN1;chr14-55646972-55647007-55641691-55641760-55648024-55648088__0:0:0,GCTTCGACACCGAGCTCGATCTATTTAATAACCCTAAACAATTTTC...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...,0000000000000000000000000000000000000000000000...


In [12]:
import pandas as pd

output_index_offset = []
output_sequence = []
# Take the barcode filtered and generate all possible single base mutation sequences. 
for i in range(len(Barcode_filtered)):
    barcode_seq = Barcode_filtered.iloc[i]['padded_sequence']
    barcode_index = Barcode_filtered.iloc[i].name
    # Generate all possible single base mutation sequences
    for j in range(len(barcode_seq)):
        for base in ['A', 'T', 'C', 'G']:
            if barcode_seq[j] != base:
                new_barcode_seq = barcode_seq
                new_barcode_seq = new_barcode_seq[:j] + base + new_barcode_seq[j+1:]
                new_barcode_index = f"{barcode_index}__initial{barcode_seq[j]}__mutated{j+1}{base}"
                output_index_offset.append(new_barcode_index)
                output_sequence.append(new_barcode_seq)
output_index_offset = pd.Series(output_index_offset)
output_sequence = pd.Series(output_sequence)

# Write to a csv file
Barcode_single_mut = pd.DataFrame({'index_offset': output_index_offset, 'padded_sequence': output_sequence})
Barcode_single_mut.to_csv("~/Downloads/shortlist_satmut_parent_seq_single_base_mutation.csv", index=False)
Barcode = Barcode_single_mut.set_index("index_offset")

# # Display all rows and columns in the notebook output
# with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
#     display(Barcode_single_mut)
Barcode

,padded_sequence
index_offset,
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0__initialC__mutated1A,ATTCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0__initialC__mutated1T,TTTCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0__initialC__mutated1G,GTTCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0__initialT__mutated2A,CATCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...
ENSG00000028203.18;VEZT;chr12-95258244-95258282-95257149-95257239-95262905-95263081__0:0:0__initialT__mutated2C,CCTCGACACCGAGCTCGATCCAAATTGTCAAGTTCTTTGTTCAGGG...
...,...
ENSG00000286034.1;RP11-654O1.1;chr8-128324908-128325005-128323130-128323188-128336131-128336213__0:0:0__initialT__mutated249C,GCTTCGACACCGAGCTCGGCCTCTTCCAGATCAAGTCCAGACTCTT...
ENSG00000286034.1;RP11-654O1.1;chr8-128324908-128325005-128323130-128323188-128336131-128336213__0:0:0__initialT__mutated249G,GCTTCGACACCGAGCTCGGCCTCTTCCAGATCAAGTCCAGACTCTT...
ENSG00000286034.1;RP11-654O1.1;chr8-128324908-128325005-128323130-128323188-128336131-128336213__0:0:0__initialC__mutated250A,GCTTCGACACCGAGCTCGGCCTCTTCCAGATCAAGTCCAGACTCTT...


In [13]:
# make index as the barcode
PSI = PSI.set_index("index_offset")
PSI = PSI.drop(columns=["Unnamed: 0"])
PSI

,769P,786O,8MGBA,A172,A375,ACHN,CAL120,COGN278,COLO783,DAOY,...,SF126,SKNAS,SNU398,SNU423,SNU449,SNUC4,T47D,TOV21G,U251MG,VMRCRCZ
index_offset,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003.15;TSPAN6;chrX-100632484-100632568-100630758-100630866-100633404-100633539__0:0:0,5.855052,6.955650,4.786596,5.066832,4.343257,5.347252,6.247928,5.172890,9.967226,8.377934,...,9.967226,4.703436,5.544321,3.755662,9.967226,9.967226,6.139551,9.967226,9.967226,9.967226
ENSG00000000003.15;TSPAN6;chrX-100633930-100634029-100632484-100632568-100635177-100635252__0:0:0,4.160823,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,...,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226,9.967226
ENSG00000000003.15;TSPAN6;chrX-100635177-100635252-100633930-100634029-100635557-100635746__0:0:0,NaN,1.069751,-0.060542,-0.147135,-0.503730,1.844684,NaN,1.994607,9.967226,2.121991,...,-0.095157,0.997839,1.779734,NaN,-1.286800,1.089583,-0.147135,0.678072,0.617465,0.828326
ENSG00000000419.14;DPM1;chr20-50945736-50945762-50942030-50942126-50945846-50945861__0:0:0,1.252026,1.712718,3.996940,2.492914,-0.233995,2.835563,1.672780,4.786596,0.158698,1.149682,...,2.632603,4.829909,1.633412,4.160823,2.093702,2.248800,3.801190,1.163165,0.794913,1.641242
ENSG00000000419.14;DPM1;chr20-50948628-50948662-50945736-50945923-50955185-50955285__0:0:0,NaN,NaN,0.280483,NaN,-3.321928,-1.581126,NaN,NaN,NaN,NaN,...,NaN,-1.116179,NaN,NaN,NaN,NaN,-1.736966,-2.248800,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000288710.1;RP11-386G11.12;chr12-49005461-49005543-49005305-49005364-49005742-49005852__0:0:0,-0.684164,-0.135578,-0.474069,-2.756957,-2.731449,0.118249,-0.714717,0.503730,0.991361,0.292127,...,-0.895303,-0.819826,0.014413,-1.203909,-2.862496,-0.794913,-2.822237,-2.822237,-0.971986,-0.129800
ENSG00000288710.1;RP11-386G11.12;chr12-49022279-49022353-49022042-49022151-49022589-49022875__0:0:0,-0.539463,0.895303,0.095157,-0.391524,-1.238212,1.116179,-0.828326,1.192349,-0.072078,0.907996,...,1.328998,-0.060542,1.581126,-0.379787,-1.422312,0.462233,-0.901645,-0.491853,0.690262,0.456320
ENSG00000288717.1;RP11-852E15.4;chr3-46000912-46000955-45995957-45996035-46001083-46001283__0:0:0,-0.309611,0.545434,0.426815,1.036911,-1.541097,0.521577,-0.228193,1.176697,0.170265,2.681408,...,0.665905,-0.158698,0.303781,-0.112475,-0.474069,0.770115,0.257222,0.794913,0.344648,2.142811


In [14]:
num_na = PSI.isna().sum().sum()
num_non_na = PSI.size - num_na

print(f"NA: {num_na}")
print(f"~NA: {num_non_na}")

NA: 119406
~NA: 1982178


In [15]:
def onehot_encode_sequences(sequences):
    """
    One-hot encodes a list of DNA sequences.

    Args:
        sequences (list of str): A list of DNA sequences, each containing characters 'A', 'G', 'C', 'T'.

    Returns:
        np.ndarray: A 3D numpy array where each sequence is one-hot encoded along the last axis.
    """
    # Define a mapping for bases
    base_mapping = {'A': 0, 'G': 1, 'C': 2, 'T': 3, 'N': 4}
    n_sequences = len(sequences)
    sequence_length = len(sequences[0]) if sequences else 0
    
    # Initialize the one-hot encoded array
    onehot_array = np.zeros((n_sequences, sequence_length, 4), dtype=int)
    
    for i, seq in enumerate(sequences):
        for j, base in enumerate(seq):
            if base in base_mapping:  # Ensure valid base
                onehot_array[i, j, base_mapping[base]] = 1
                
    return onehot_array

Barcode_onehot = onehot_encode_sequences(list(Barcode['padded_sequence'].values))
# Barcode_onehot = np.concatenate((Barcode_onehot, twist_array), axis=-1)
print(Barcode_onehot.shape)
Barcode_dict = dict(zip(Barcode.index, Barcode_onehot))


(25500, 250, 4)


In [16]:
class PSIDataset(Dataset):
    def __init__(self, PSI_df, Barcode_dict, GX_dict):
        self.samples = []

        for barcode in Barcode_dict.keys():
            barcode_array = Barcode_dict[barcode]

            for celltype in ["8MGBA", "HCC1428", "KELLY", "KMRC20", "HCC38", "JHH6", "MCF7", "T47D"]:
                gx_array = GX_dict[celltype]     
                self.samples.append((celltype, barcode, barcode_array, gx_array, 0))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        celltype, barcode, barcode_array, gx_array, psi_value = self.samples[idx]
        return celltype, barcode, barcode_array, gx_array, psi_value
    
def split_dataset_by_barcode(dataset, test_size=0.2, random_state=42):
    """
    Split the dataset into train and test subsets based on barcode.
    Params:
    - dataset: PSIDataset
    - test_size: ratio
    - random_state
    
    Return:
    - train_dataset
    - test_dataset
    """
    # Extract all barcodes
    barcodes = np.array([sample[1] for sample in dataset.samples])
    
    # Get unique barcodes and split
    unique_barcodes = np.unique(barcodes)
    
    train_barcodes, test_barcodes = train_test_split(
        unique_barcodes, test_size=test_size, random_state=random_state
    )
    
    print(f"Train barcodes: {train_barcodes[:5]}")
    print(f"Test barcodes: {test_barcodes[:5]}") 
    
    train_barcodes_set = set(train_barcodes)
    test_barcodes_set = set(test_barcodes)
    
    # Get indices for train and test
    train_indices = np.where(np.isin(barcodes, list(train_barcodes_set)))[0]
    test_indices = np.where(np.isin(barcodes, list(test_barcodes_set)))[0]
    print(f"Train indices: {len(train_indices)}, Test indices: {len(test_indices)}")
    
    # Construct Subsets
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    return train_dataset, test_dataset

def split_dataset_by_celltype(dataset, test_size=0.2, random_state=42):
    """
    Split the dataset into train and test subsets based on celltype.
    Params:
    - dataset: PSIDataset
    - test_size: ratio
    - random_state
    
    Return:
    - train_dataset
    - test_dataset
    """
    celltypes = np.array([sample[0] for sample in dataset.samples])
    
    unique_celltypes = np.unique(celltypes)
    train_celltypes, test_celltypes = train_test_split(
        unique_celltypes, test_size=test_size, random_state=random_state
    )
    
    print(f"Train celltypes: {len(train_celltypes)}, Test celltypes: {len(test_celltypes)}")
    
    train_indices = [i for i, celltype in enumerate(celltypes) if celltype in train_celltypes]
    test_indices = [i for i, celltype in enumerate(celltypes) if celltype in test_celltypes]
    
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    assert len(train_dataset) == len(train_indices), "Train dataset size mismatch!"
    assert len(test_dataset) == len(test_indices), "Test dataset size mismatch!"
    
    return train_dataset, test_dataset

def split_dataset_random(dataset, test_size=0.2, random_state=42):

    dataset_size = len(dataset)
    
    indices = list(range(dataset_size))
    
    train_indices, test_indices = train_test_split(
        indices, test_size=test_size, random_state=random_state
    )
    
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)
    
    return train_dataset, test_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

lr = 1e-4
epochs = 5
batch_size = 1024

dataset = PSIDataset(PSI, Barcode_dict, GX_dict)
dataset_size = len(dataset)
print(f"Total samples: {dataset_size}")

# # train_dataset, test_dataset = split_dataset_random(dataset, test_size=0.2, random_state=42)
# train_dataset, test_dataset = split_dataset_by_barcode(dataset, test_size=0.2, random_state=42)

# # train_dataset, test_dataset = split_dataset_by_celltype(dataset, test_size=0.2, random_state=42)
# print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8)

Total samples: 204000


## Convolutional Neural Network

In [17]:
class GexFullyConnected(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(in_features=19221, out_features=1024)
        self.fc2 = nn.Linear(in_features=1024, out_features=256)
        self.fc3 = nn.Linear(in_features=256, out_features=128)
        self.BN1 = nn.BatchNorm1d(1024)
        self.BN2 = nn.BatchNorm1d(256) 
        self.dropout = nn.Dropout(0.2)     

    def forward(self, x):
        x = F.relu(self.BN1(self.fc1(x)))
        x = F.relu(self.BN2(self.fc2(x)))
        x = F.relu(self.fc3(x))
        return x

class ResNetCNN(nn.Module):
    def __init__(self, channels, dropout=0.2):
        super().__init__() 
        
        self.conv1_5 =  nn.Conv1d(channels, channels, kernel_size=5, padding=2)  
        self.conv1_11 = nn.Conv1d(channels, channels, kernel_size=11, padding=5)
        self.conv1_21 = nn.Conv1d(channels, channels, kernel_size=21, padding=10)

        self.batchnorm1 = nn.BatchNorm1d(channels * 3) 
        self.dropout = nn.Dropout(dropout)
        self.conv_merge = nn.Conv1d(channels * 3, channels, kernel_size=1)

        self.conv2 = nn.Conv1d(channels, channels, kernel_size=5, padding=2)
        self.batchnorm2 = nn.BatchNorm1d(channels)

    def forward(self, x):
        residual = x 
        x1 = self.conv1_5(x)
        x2 = self.conv1_11(x)
        x3 = self.conv1_21(x)

        x = torch.cat([x1, x2, x3], dim=1) 

        x = self.dropout(F.relu(self.batchnorm1(x)))
        x = self.conv_merge(x) 
        x = self.dropout(F.relu(self.batchnorm2(self.conv2(x))))
        
        x += residual  
        return x

class SequenceCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.dropout = nn.Dropout(0.2)
        
        self.conv1 = nn.Conv1d(in_channels=4, out_channels=128, kernel_size=1, padding='same')
        self.batchnorm1 = nn.BatchNorm1d(128)
    
        self.resnetCNN1 = nn.Sequential(
            ResNetCNN(128),
            ResNetCNN(128),
            ResNetCNN(128),
        )
        
        self.maxpool = nn.MaxPool1d(kernel_size=2)
        
        self.conv2 = nn.Conv1d(in_channels=128, out_channels=64, kernel_size=1, padding='same')
        self.batchnorm2 = nn.BatchNorm1d(64)
        
        self.resnetCNN2 = nn.Sequential(
            ResNetCNN(64),
            ResNetCNN(64),
            ResNetCNN(64),
        )
        
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=16, kernel_size=1, padding='same')
        self.batchnorm3 = nn.BatchNorm1d(16)
        
        self.resnetCNN3 = nn.Sequential(
            ResNetCNN(16),
            ResNetCNN(16),
            ResNetCNN(16),
        )

        self.conv4 = nn.Conv1d(in_channels=128, out_channels=16, kernel_size=1, padding='same')
        self.maxpool8 = nn.MaxPool1d(kernel_size=8)
        
        # Linear layers
        self.fc1 = nn.Linear(in_features=496, out_features=128)
        
    def forward(self, x):
        x = self.dropout(F.relu(self.batchnorm1(self.conv1(x))))
        residual = x
        x_residual = x
        x = self.resnetCNN1(x)
        x += residual
        x = self.maxpool(x)
        
        x = self.dropout(F.relu(self.batchnorm2(self.conv2(x))))
        residual = x
        x = self.resnetCNN2(x)
        x += residual
        x = self.maxpool(x)
        
        x = self.dropout(F.relu(self.batchnorm3(self.conv3(x))))
        residual = x
        x = self.resnetCNN3(x)
        x += residual
        x = self.maxpool(x)
        
        x_residual = self.conv4(x_residual)
        x_residual = self.maxpool8(x_residual)
        x = x + x_residual
        
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        
        return x

class Predictor_gene_barcode(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Barcode_model = SequenceCNN()
        self.GX_model = GexFullyConnected()

        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(in_features=64, out_features=1)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, barcode, gx, condition):
        
        z_barcode = self.Barcode_model(barcode)
        
        z_gx = self.GX_model(gx)
        x = z_barcode + z_gx + condition
        x = self.dropout(F.leaky_relu(self.bn1(self.fc1(x))))
        x = self.fc2(x)
        
        return x
    
class Predictor_barcode(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.Barcode_model = SequenceCNN()

        self.fc1 = nn.Linear(in_features=128, out_features=64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(in_features=64, out_features=1)
        
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, barcode):
        
        x = self.Barcode_model(barcode)
        x = self.dropout(F.leaky_relu(self.bn1(self.fc1(x))))
        x = self.fc2(x)
        
        return x

# Evaluation for model_barcode

In [18]:
import matplotlib as mpl
import os

mpl.rcParams['pdf.fonttype'] = 42

output_dir = "/home/dawn/lab_projects/splicing/for_kai/single_mut_eval/"
# Make the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
seed_set = range(10)
for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=8)
    
    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    MSELoss = nn.MSELoss()
    optimizer = torch.optim.Adam(model_barcode.parameters(), lr=lr)

    predict_psi = []
    real_psi = []
    celltypes = []
    barcode_names = []
    with torch.no_grad():
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            psi = psi.to(device).float()  # Ensure psi is on the same device as barcode/model

            psi_pretrain = model_barcode(barcode)
            predict_psi.extend(psi_pretrain.squeeze().cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    print(f"Seed {seed}:")

    # Create a DataFrame to store the predictions and actual values for this seed
    predictions_df = pd.DataFrame({
        'Barcode': barcode_names,
        'Celltype': celltypes,
        'Predicted_PSI': predict_psi
    })

    # Save predictions to CSV for this seed
    predictions_csv_path = os.path.join(output_dir, f'single_base_mut_predictions_barcode_seed{seed}.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    print(f"Saved predictions to {predictions_csv_path}")

Seed 0:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed0.csv
Seed 1:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed1.csv
Seed 2:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed2.csv
Seed 3:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed3.csv
Seed 4:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed4.csv
Seed 5:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed5.csv
Seed 6:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_barcode_seed6.csv
Seed 7:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single

In [19]:
# Also eval for gene_barcode

import matplotlib as mpl
import os

mpl.rcParams['pdf.fonttype'] = 42

output_dir = "/home/dawn/lab_projects/splicing/for_kai/single_mut_eval/"
# Make the output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
seed_set = range(10)
for seed in seed_set:
    random.seed(seed)
    np.random.seed(seed)    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=8)
    
    model_barcode = Predictor_barcode()
    model_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_barcode.eval()
    model_barcode = model_barcode.to(device)

    MSELoss = nn.MSELoss()
    optimizer = torch.optim.Adam(model_barcode.parameters(), lr=lr)


    model_gene_barcode = Predictor_gene_barcode()
    model_gene_barcode.load_state_dict(torch.load(f"/home/dawn/lab_projects/splicing/for_kai/no_mutagenesis_data/model_barcode_gene_PSI_no_mut_5epoch_seed{seed}.pth"))
    model_gene_barcode.eval()
    model_gene_barcode = model_gene_barcode.to(device)

    predict_psi = []
    real_psi = []
    psi_residuals = []
    celltypes = []
    barcode_names = []
    
    with torch.no_grad():
        recon_loss = 0
        for i, (celltype, barcode_name, barcode, gx, psi) in enumerate(test_loader):
            barcode = barcode.to(device).float()
            barcode = barcode.permute(0, 2, 1)
            gx = gx.to(device).float()
            psi = psi.to(device).float()
        
            condition = model_barcode.Barcode_model(barcode)
            psi_pretrain = model_barcode(barcode)
        
            psi_residual = model_gene_barcode(barcode, gx, condition)
            
            predict_psi.extend(psi_residual.squeeze().cpu().numpy()+psi_pretrain.squeeze().cpu().numpy())
            celltypes.extend(celltype)
            barcode_names.extend(barcode_name)

    print(f"Seed {seed}:")

    # Create a DataFrame to store the predictions and actual values for this seed
    predictions_df = pd.DataFrame({
        'Barcode': barcode_names,
        'Celltype': celltypes,
        'Predicted_PSI': predict_psi
    })

    # Save predictions to CSV for this seed
    predictions_csv_path = os.path.join(output_dir, f'single_base_mut_predictions_gene_barcode_seed{seed}.csv')
    predictions_df.to_csv(predictions_csv_path, index=False)
    print(f"Saved predictions to {predictions_csv_path}")

Seed 0:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed0.csv
Seed 1:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed1.csv
Seed 2:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed2.csv
Seed 3:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed3.csv
Seed 4:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed4.csv
Seed 5:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed5.csv
Seed 6:
Saved predictions to /home/dawn/lab_projects/splicing/for_kai/single_mut_eval/single_base_mut_predictions_gene_barcode_seed6.csv
Seed 7:
Saved predictions to /home/dawn/l